In [ ]:
from pathlib import Path
import pandas as pd
import shutil

In [ ]:
# input folder
input_dir = Path("/Users/miasmacbook/Desktop/6-months_project/data/disease_gene_table")

# output base folder
output_base = Path("/Users/miasmacbook/Desktop/6-months_project/data/disease_gene_table")
output_base.mkdir(parents=True, exist_ok=True)

summary_rows = []
error_rows = []

files = sorted(input_dir.glob("*.csv"))
print(f"Found {len(files)} csv files")


In [ ]:
for fpath in files:
    try:
        df = pd.read_csv(fpath)

        if "targetId" not in df.columns:
            print(f"Skipping {fpath.name}: no targetId column")
            error_rows.append([fpath.name, "no targetId column"])
            continue

        # uniq targetId first
        unique_genes = (
            df["targetId"]
            .dropna()
            .astype(str)
            .str.strip()
        )
        unique_genes = unique_genes[unique_genes != ""].drop_duplicates()

        n_genes = len(unique_genes)

        # folder name directly by gene count
        folder_name = f"{n_genes}_gene_related_disease"
        out_dir = output_base / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)

        shutil.move(str(fpath), str(out_dir / fpath.name))
        summary_rows.append({
            "file_name": fpath.name,
            "n_unique_genes": n_genes,
            "output_folder": folder_name
        })

    except Exception as e:
        print(f"Error processing {fpath.name}: {e}")
        error_rows.append([fpath.name, str(e)])

# save summary
summary_df = pd.DataFrame(summary_rows)
summary_out = output_base / "disease_file_grouping_summary.csv"
summary_df.to_csv(summary_out, index=False)

print("\nDone!")
print(f"Processed files: {len(summary_df)}")
print(f"Summary saved to: {summary_out}")

# save errors
if error_rows:
    error_df = pd.DataFrame(error_rows, columns=["file_name", "error"])
    error_out = output_base / "disease_file_grouping_errors.csv"
    error_df.to_csv(error_out, index=False)
    print(f"Errors saved to: {error_out}")